In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("Housing.csv")

print("Dataset Shape:", df.shape)
print(df.head())

Dataset Shape: (20640, 10)
   longitude  latitude  housing_median_age  total_rooms  total_bedrooms  \
0    -122.23     37.88                41.0        880.0           129.0   
1    -122.22     37.86                21.0       7099.0          1106.0   
2    -122.24     37.85                52.0       1467.0           190.0   
3    -122.25     37.85                52.0       1274.0           235.0   
4    -122.25     37.85                52.0       1627.0           280.0   

   population  households  median_income  median_house_value ocean_proximity  
0       322.0       126.0         8.3252            452600.0        NEAR BAY  
1      2401.0      1138.0         8.3014            358500.0        NEAR BAY  
2       496.0       177.0         7.2574            352100.0        NEAR BAY  
3       558.0       219.0         5.6431            341300.0        NEAR BAY  
4       565.0       259.0         3.8462            342200.0        NEAR BAY  


In [3]:
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  float64
 9   ocean_proximity     20640 non-null  object 
dtypes: float64(9), object(1)
memory usage: 1.6+ MB


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
count,20640.000000,20640.000000,20640.000000,20640.000000,20433.000000,20640.000000,20640.000000,20640.000000,20640.000000
mean,-119.569704,35.631861,28.639486,2635.763081,537.870553,1425.476744,499.539680,3.870671,206855.816909
std,2.003532,2.135952,12.585558,2181.615252,421.385070,1132.462122,382.329753,1.899822,115395.615874
min,-124.350000,32.540000,1.000000,2.000000,1.000000,3.000000,1.000000,0.499900,14999.000000
25%,-121.800000,33.930000,18.000000,1447.750000,296.000000,787.000000,280.000000,2.563400,119600.000000
50%,-118.490000,34.260000,29.000000,2127.000000,435.000000,1166.000000,409.000000,3.534800,179700.000000
75%,-118.010000,37.710000,37.000000,3148.000000,647.000000,1725.000000,605.000000,4.743250,264725.000000
max,-114.310000,41.950000,52.000000,39320.000000,6445.000000,35682.000000,6082.000000,15.000100,500001.000000


In [4]:
df.isnull().sum()

longitude               0
latitude                0
housing_median_age      0
total_rooms             0
total_bedrooms        207
population              0
households              0
median_income           0
median_house_value      0
ocean_proximity         0
dtype: int64

In [5]:
df = df.dropna()

In [6]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['ocean_proximity'] = le.fit_transform(df['ocean_proximity'])

In [7]:
x = df.drop('median_house_value', axis=1)
y = df['median_house_value'] 

In [8]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

x_scaled = scaler.fit_transform(x)

In [9]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test=train_test_split(x_scaled,y,
                                                 test_size=0.2,
                                                 random_state=42)

In [10]:
from sklearn.linear_model import LinearRegression
lr = LinearRegression()

lr.fit(x_train, y_train)
lr_pred = lr.predict(x_test)

from sklearn.metrics import mean_squared_error, r2_score
lr_rmse = mean_squared_error(y_test, lr_pred) ** 0.5
lr_r2 = r2_score(y_test, lr_pred)

print("RMSE:", lr_rmse)
print("R2 Score:", lr_r2)
lr

RMSE: 70171.99539639697
R2 Score: 0.6399236679243399


,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [11]:
from sklearn.linear_model import Ridge

ridge = Ridge(alpha=1.0)

ridge.fit(x_train, y_train)

ridge_pred = ridge.predict(x_test)

ridge_rmse = np.sqrt(mean_squared_error(y_test, ridge_pred))
ridge_r2 = r2_score(y_test, ridge_pred)

print("\nRidge Regression")
print("RMSE:", ridge_rmse)
print("R²:", ridge_r2)


Ridge Regression
RMSE: 70172.30149967065
R²: 0.6399205264778467


In [12]:
from sklearn.tree import DecisionTreeRegressor

tree = DecisionTreeRegressor(random_state=42)

tree.fit(x_train, y_train)

train_pred = tree.predict(x_train)
test_pred = tree.predict(x_test)

train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))

print("\nDecision Tree Overfitting Analysis")
print("Train RMSE:", train_rmse)
print("Test RMSE:", test_rmse)



Decision Tree Overfitting Analysis
Train RMSE: 0.0
Test RMSE: 68497.06644960509


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV

rf = RandomForestRegressor(random_state=42)

param_grid_rf = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5]
}

rf_grid = GridSearchCV(
    rf,
    param_grid_rf,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

rf_grid.fit(x_train, y_train)

best_rf = rf_grid.best_estimator_

rf_pred = best_rf.predict(x_test)

rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_r2 = r2_score(y_test, rf_pred)

print("Random Forest RMSE:", rf_rmse)
print("Random Forest R²:", rf_r2)

In [ ]:
from sklearn.model_selection import cross_val_score

lr_cv = cross_val_score(
    lr,
    x_scaled,
    y,
    cv=5,
    scoring='r2'
).mean()

ridge_cv = cross_val_score(
    ridge,
    x_scaled,
    y,
    cv=5,
    scoring='r2'
).mean()

tree_cv = cross_val_score(
    tree,
    x_scaled,
    y,
    cv=5,
    scoring='r2'
).mean()

print("\nCross Validation Scores")
print("Linear Regression CV:", lr_cv)
print("Ridge Regression CV:", ridge_cv)
print("Decision Tree CV:", tree_cv)

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "max_depth": [3, 5, 7, 10],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

grid = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

grid.fit(x_train, y_train)

print("\nBest Parameters")
print(grid.best_params_)

In [ ]:
best_tree = grid.best_estimator_

tree_pred = best_tree.predict(x_test)

tree_rmse = np.sqrt(mean_squared_error(y_test, tree_pred))
tree_r2 = r2_score(y_test, tree_pred)

print("\nTuned Decision Tree")
print("RMSE:", tree_rmse)
print("R²:", tree_r2)

In [ ]:
best_tree_cv = cross_val_score(
    best_tree,
    x_scaled,
    y,
    cv=5,
    scoring='r2'
).mean()

print("Tuned Decision Tree CV:", best_tree_cv)

In [ ]:
results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Ridge Regression",
        "Tuned Decision Tree"
    ],
    "RMSE": [
        lr_rmse,
        ridge_rmse,
        tree_rmse
    ],
    "R2 Score": [
        lr_r2,
        ridge_r2,
        tree_r2
    ],
    "CV Score": [
        lr_cv,
        ridge_cv,
        best_tree_cv
    ]
})

print("\nModel Comparison")
print(results)

In [ ]:
best_model = results.loc[
    results["R2 Score"].idxmax()
]

print("\nBest Model:")
print(best_model)

In [ ]:
import joblib
joblib.dump(lr_model, "model.pkl")
joblib.dump(vector, "tfidf.pkl")